In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load raw file
raw_df = pd.read_parquet("data/raw/census/census_raw_all_geographies.parquet")
print(f"   Rows    : {len(raw_df)}")
print(f"   Columns : {len(raw_df.columns)}")
print(f"\nGeographies:")
for geo in raw_df["geography"].unique():
    years = sorted(raw_df[raw_df["geography"]==geo]["acs_year"].unique())
    if len(years) > 3:
        print(f"   {geo}: {years[0]} → {years[-1]} ({len(years)} years)")
    else:
        print(f"   {geo}: {years}")

   Rows    : 17
   Columns : 51

Geographies:
 Tehama County (2010)
 Tehama County (2011)
 Tehama County (2012)
 Tehama County (2013)
 Tehama County (2014)
 Tehama County (2015)
 Tehama County (2016)
 Tehama County (2017)
 Tehama County (2018)
 Tehama County (2019)
 Tehama County (2020)
 Tehama County (2021)
 Tehama County (2022)
 Tehama County (2023)
 Tehama County (2024)
 California (2024)
 United States (2024)


In [2]:
# Current year = 2024, Prior year = 2023
current = raw_df[raw_df["acs_year"] == 2024].copy()
prior   = raw_df[raw_df["acs_year"] == 2023].copy()

In [3]:
# Calculate Rates
# Takes raw counts and turns them into meaningful percentages
# We do this for ALL 3 geographies so comparisons work correctly

def calculate_rates(df):
    """
    Add calculated rate columns to a dataframe.
    Works for any geography (Tehama, CA, US).
    """
    d = df.copy()
    pop = d["total_population"]

    # ── Age & Sex 
    d["male_pct"]   = (d["male_population"]   / pop * 100).round(1)
    d["female_pct"] = (d["female_population"] / pop * 100).round(1)

    d["pop_under_5"] = d["male_under_5"] + d["female_under_5"]
    d["pop_under_5_pct"] = (d["pop_under_5"] / pop * 100).round(1)

    d["pop_under_18"] = (
        d["male_under_5"]   + d["female_under_5"]  +
        d["male_5_to_9"]    + d["female_5_to_9"]   +
        d["male_10_to_14"]  + d["female_10_to_14"] +
        d["male_15_to_17"]  + d["female_15_to_17"]
    )
    d["pop_under_18_pct"] = (d["pop_under_18"] / pop * 100).round(1)

    d["pop_over_65"] = (
        d["male_65_66"]    + d["female_65_66"]  +
        d["male_67_69"]    + d["female_67_69"]  +
        d["male_70_74"]    + d["female_70_74"]  +
        d["male_75_79"]    + d["female_75_79"]  +
        d["male_80_84"]    + d["female_80_84"]  +
        d["male_85_plus"]  + d["female_85_plus"]
    )
    d["pop_over_65_pct"] = (d["pop_over_65"] / pop * 100).round(1)

    # ── Race & Ethnicity
    d["hispanic_pct"]        = (d["hispanic_latino"] / pop * 100).round(1)
    d["white_pct"]           = (d["white_alone"]     / pop * 100).round(1)
    d["black_pct"]           = (d["black"]           / pop * 100).round(1)
    d["aian_pct"]            = (d["american indian_alone"]    / pop * 100).round(1)
    d["asian_pct"]           = (d["asian_alone"]     / pop * 100).round(1)
    d["nhpi_pct"]            = (d["nhpi_alone"]      / pop * 100).round(1)
    d["two_or_more_pct"]     = (d["two_or_more_races"]/ pop * 100).round(1)

    # ── Income & Poverty
    d["poverty_rate_pct"] = (
        d["population_below_poverty"] / 
        d["poverty_universe"] * 100
    ).round(1)

    # ── Housing
    d["homeownership_rate_pct"] = (
        d["owner_occupied_units"] / 
        d["occupied_housing_units"] * 100
    ).round(1)

    return d


# Apply to both years
current_calc = calculate_rates(current)
prior_calc   = calculate_rates(prior)

print("Rates calculated for all geographies")
print("\nTehama County 2024:")
tehama = current_calc[current_calc["geography"] == "Tehama County"].iloc[0]
print(f"  Male %              : {tehama['male_pct']}%")
print(f"  Female %            : {tehama['female_pct']}%")
print(f"  Population under 5  : {tehama['pop_under_5_pct']}%")
print(f"  Population under 18 : {tehama['pop_under_18_pct']}%")
print(f"  Population over 65  : {tehama['pop_over_65_pct']}%")
print(f"  Hispanic %          : {tehama['hispanic_pct']}%")
print(f"  White %             : {tehama['white_pct']}%")
print(f"  Poverty rate        : {tehama['poverty_rate_pct']}%")
print(f"  Homeownership rate  : {tehama['homeownership_rate_pct']}%")

Rates calculated for all geographies

Tehama County 2024:
  Male %              : 49.7%
  Female %            : 50.3%
  Population under 5  : 5.8%
  Population under 18 : 24.1%
  Population over 65  : 20.1%
  Hispanic %          : 29.1%
  White %             : 66.7%
  Poverty rate        : 15.6%
  Homeownership rate  : 67.1%


In [4]:
# Build Comparison Table 

def get_geo(df, geo):
    """Extract one geography row as a Series."""
    return df[df["geography"] == geo].iloc[0]

# ── Get all 6 rows 
tehama_now  = get_geo(current_calc, "Tehama County")
ca_now      = get_geo(current_calc, "California")
us_now      = get_geo(current_calc, "United States")
tehama_prior= get_geo(prior_calc,   "Tehama County")

# ── Define indicators to compare 
indicators = [
    # label                    column                      unit   higher_is_better
    ("Total Population",       "total_population",         "n",   True),
    ("Median Age",             "median_age",               "yrs", False),
    ("Male %",                 "male_pct",                 "%",   False),
    ("Female %",               "female_pct",               "%",   False),
    ("Population Under 5 %",   "pop_under_5_pct",          "%",   False),
    ("Population Under 18 %",  "pop_under_18_pct",         "%",   False),
    ("Population Over 65 %",   "pop_over_65_pct",          "%",   False),
    ("Hispanic %",             "hispanic_pct",             "%",   False),
    ("White %",                "white_pct",                "%",   False),
    ("Black %",                "black_pct",                "%",   False),
    ("AIAN %",                 "aian_pct",                 "%",   False),
    ("Asian %",                "asian_pct",                "%",   False),
    ("NHPI %",                 "nhpi_pct",                 "%",   False),
    ("Two or More Races %",    "two_or_more_pct",          "%",   False),
    ("Median HH Income",       "median_household_income",  "$",   True),
    ("Per Capita Income",      "per_capita_income",        "$",   True),
    ("Poverty Rate %",         "poverty_rate_pct",         "%",   False),
    ("Median Home Value",      "median_home_value",        "$",   True),
    ("Median Costs w/Mortgage","median_costs_with_mortgage","$",  False),
    ("Median Costs w/o Mortgage","median_costs_without_mortgage","$",False),
    ("Median Gross Rent",      "median_gross_rent",        "$",   False),
    ("Homeownership Rate %",   "homeownership_rate_pct",   "%",   True),
    ("Housing Units",          "housing_units",            "n",   False),
]

# ── Build rows 
rows = []
for label, col, unit, higher_is_better in indicators:
    try:
        t_now   = tehama_now[col]
        t_prior = tehama_prior[col]
        ca      = ca_now[col]
        us      = us_now[col]

        # vs CA and vs US
        vs_ca = round(t_now - ca, 1)   if unit != "n" else None
        vs_us = round(t_now - us, 1)   if unit != "n" else None

        # Prior year change
        change = round(t_now - t_prior, 1) if unit != "n" else round(t_now - t_prior, 0)

        # Trend direction
        if unit == "n":
            trend = "—"
        elif abs(change) < 0.5:
            trend = "→ Stable"
        elif (change > 0 and higher_is_better) or (change < 0 and not higher_is_better):
            trend = "↑ Better"
        else:
            trend = "↓ Worse"

        rows.append({
            "Indicator":    label,
            "Tehama 2024":  t_now,
            "CA 2024":      ca,
            "US 2024":      us,
            "Tehama 2023":  t_prior,
            "vs CA":        vs_ca,
            "vs US":        vs_us,
            "Change":       change,
            "Trend":        trend,
            "Unit":         unit,
        })

    except Exception as e:
        print(f"Skipped {label}: {e}")

# ── Create DataFrame 
comparison_df = pd.DataFrame(rows)

print("Comparison table")
print(f"   Rows : {len(comparison_df)}")
print(f"\nPreview:")
print(comparison_df[["Indicator","Tehama 2024","CA 2024","US 2024","Trend"]].to_string(index=False))

Comparison table
   Rows : 23

Preview:
                Indicator  Tehama 2024    CA 2024     US 2024    Trend
         Total Population      65167.0 39287377.0 334922499.0        —
               Median Age         40.0       37.9        38.9 → Stable
                   Male %         49.7       49.9        49.5 → Stable
                 Female %         50.3       50.1        50.5 → Stable
     Population Under 5 %          5.8        5.5         5.6 → Stable
    Population Under 18 %         24.1       22.0        22.0 → Stable
     Population Over 65 %         20.1       15.7        17.2 → Stable
               Hispanic %         29.1       40.2        19.3  ↓ Worse
                  White %         66.7       39.7        61.0 ↑ Better
                  Black %          0.9        5.4        12.2 → Stable
                   AIAN %          2.0        1.3         0.9 → Stable
                  Asian %          2.0       15.5         6.0 → Stable
                   NHPI %          0.

In [5]:
print(comparison_df.columns.tolist())

['Indicator', 'Tehama 2024', 'CA 2024', 'US 2024', 'Tehama 2023', 'vs CA', 'vs US', 'Change', 'Trend', 'Unit']


In [6]:
print(raw_df.shape)
print(raw_df["geography"].unique())
print(raw_df["acs_year"].unique())

(17, 51)
<ArrowStringArray>
['Tehama County', 'California', 'United States']
Length: 3, dtype: str
[2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023
 2024]
